# Lecture 22: Backfills

**DSC 592 · Loading the past with the pipeline you already have**

Your hourly job only collects weather **from the moment it starts**. A **backfill** loads data for a period **in the past**.

| When | Example |
| --- | --- |
| A new pipeline needs history | The dashboard wants last week, but the job started this week |
| After an outage | The job was paused from 2 to 6 a.m. |
| After a logic change | Old results must be recomputed (a full refresh) |

For your project: a trend chart, or a model (Option B), needs **history**.

## The pattern: same layout, same shape

```text
live:      fetch_weather.ipynb ──► .../2026/11/19/14/wilmington.json
backfill:  L22backfill.ipynb   ──► .../2026/11/13/03/wilmington_backfill.json
                                       │  same folders, same JSON shape
                                       ▼
          weather-ETL (no code change): bronze → silver → gold → dashboard
```

- Auto Loader just sees **new files**; it does not care who wrote them
- Silver reads `payload.current`, so each past hour goes into `payload.current`
- The `_backfill` name only marks where a file came from (visible in `source_file`)

## 1. Settings

In [0]:
import os, json
from datetime import datetime, timezone
import pandas as pd
import requests

LANDING   = "/Volumes/dsc592/weather/landing/weather"
PAST_DAYS = 7                       # how far back (the forecast API allows up to about 3 months)

# The cities in ingest/cities.csv 
# modify your path
cities = pd.read_csv("/Workspace/Users/chenxuemei@uncw.edu/weather-team-template/ingest/cities.csv")
display(cities)

## 2. Step 1: ask the API for the past

- `past_days = 7`: the 7 days **before** today, plus today: (7 + 1) × 24 hours
- The answer is the same `hourly` block as in Lecture 16: **parallel arrays**, `time[i]` goes with `temperature_2m[i]`

In [0]:
URL  = "https://api.open-meteo.com/v1/forecast"
VARS = ["temperature_2m", "relative_humidity_2m", "precipitation",
        "wind_speed_10m", "weather_code"]

def fetch_past(lat, lon):
    r = requests.get(URL, timeout=30, 
                     params={
        "latitude": lat, "longitude": lon, "hourly": ",".join(VARS),
        "past_days": PAST_DAYS, "forecast_days": 1, "timezone": "UTC"})
    r.raise_for_status()
    return r.json()

In [0]:
sample = fetch_past(cities.lat[0], cities.lon[0])

# sample["hourly"] has the data that we need
print(sample["hourly"].keys())
print(sample["hourly"]["time"][:3])
print(sample["hourly"]["temperature_2m"][:3])

`sample["hourly"]["time"][i]` is in isoformat which is the default by Open-Meteo

## 3. Step 2: one file per city per past hour

Each past hour becomes one landing file, in the folder of **the hour the data describes**, shaped like a live one: that hour's values go into `payload.current`.

- `enumerate` gives both the **position** `i` (0, 1, 2, …) and the **value** `t` ("2026-11-12T00:00"). We need `i` to pick the matching value from every other array.
- `fromisoformat(t)` turns the text into a datetime. `.replace(tzinfo=timezone.utc)` marks it as UTC (the API returned UTC because we asked for "timezone": "UTC"), so it can be compared with this_hour.


In [0]:
now       = datetime.now(timezone.utc)
this_hour = now.replace(minute=0, second=0, microsecond=0)
written = skipped = 0

for row in cities.itertuples():
    data   = fetch_past(row.lat, row.lon)
    # hourly = data["hourly"]
    slug   = row.city.lower().replace(" ", "_")

    # enumerate() gives both position and value
    for i, t in enumerate(data["hourly"]["time"]):
        ts = datetime.fromisoformat(t).replace(tzinfo=timezone.utc)
        if ts >= this_hour:
            break                                        # only hours that are over

        folder = f"{LANDING}/{ts:%Y/%m/%d/%H}"           # the data's hour, not today
        live_file   = f"{folder}/{slug}.json"                 # written by fetch_weather
        bf_file   = f"{folder}/{slug}_backfill.json"        # written by this notebook
        if os.path.exists(live_file) or os.path.exists(bf_file):   # if either file exists
            skipped += 1
            continue                                     # that hour is already covered

        current = {"time": t}            # initialze a dict with key "time"
        for v in VARS:
            current[v] = data["hourly"][v][i]                    # hour i of each array
        record = {"city": row.city,
                  "fetched_at": now.isoformat(timespec="seconds"),
                  "payload": {"latitude": data["latitude"],
                              "longitude": data["longitude"], "current": current}}
        os.makedirs(folder, exist_ok=True)
        with open(bf_file, "w") as f:
            json.dump(record, f)
        written += 1

print(written, "files written,", skipped, "skipped (already covered)")

## 4. Two rules make it safe to rerun

- **Only past hours:** stop at the current hour; the rest of the day is a forecast
- **Never overwrite:** skip an hour that already has a file for the city (live, or an earlier backfill)

**Try it:** run the cell above again. It should print **0 files written**: everything is skipped. And Auto Loader never loads a file twice.

Same idea as retries in Lecture 19: a backfill must be **idempotent**. Running it twice must not change the result.

## 5. Load it and check

1. **Run now** on `weather-hourly`
2. How far back does silver go now?

In [0]:
%sql
-- for each city, find the earliest obs_ts and number of observations


3. Dashboard: set `hours_back` to **168** (one week): the line chart now shows the whole week

## 6. Some notes

- **Two clocks again:** backfilled rows get `ingest_ts` = **now**, so `detected_at` = now. `check_heat_alert` would treat a hot hour from last week as **new**
- **Gaps:** an hour the API did not return stays empty; `n_obs` in gold shows it
- **Clean up:** `*_backfill.json` files are easy to find; delete them if you wish, then **Full refresh** `weather-ETL`